# Running GRPO Training with NVIDIA NeMo RL on Vertex AI Training Clusters

This notebook walks you through running a GRPO (Group Relative Policy Optimization) training job using the NVIDIA NeMo-RL framework on a [Vertex AI Training Cluster (VTC)](https://cloud.google.com/vertex-ai/docs/training/training-clusters/overview).

**NeMo RL** is designed to align large language models with human preferences and instructions, supporting workflows such as:
- Supervised Fine-Tuning (SFT)
- Preference-tuning (DPO — Direct Preference Optimization)
- Reinforcement Learning (GRPO, RLHF)

---

## Overview

<img src="llm-corp-blog-nemo-rl-gym-1280x680-4485700.jpg" width="480"/>

The end-to-end workflow consists of the following steps:

| Step | Description |
|------|-------------|
| 1 | [Create a Vertex AI Training Cluster](#step-1-prerequisites-create-a-cluster) |
| 2 | [Connect to the cluster login node](#step-2-connect-to-the-cluster-login-node) |
| 3 | [Set credentials (HF Token, W&B key)](#step-3-configure-authentication) |
| 4 | [Download the prebuilt container image](#step-4-download-the-container-image) |
| 5 | [Clone the NeMo RL repository](#step-5-clone-the-nemo-rl-repository) |
| 6 | [Submit the GRPO training job](#step-6-launch-the-training-job) |
| 7 | [Monitor logs and checkpoints](#step-7-monitor-the-job) |

---

## Step 1: Prerequisites — Create a Cluster

Before you begin, you need a running Vertex AI Training Cluster. Follow the official guide to provision one:

**[Create a Vertex AI Training Cluster →](https://cloud.google.com/vertex-ai/docs/training/training-clusters/create-cluster)**

Supported GPU node types and their corresponding `--cluster_type` values used later:

| Node Type | GPU | `cluster_type` value |
|-----------|-----|---------------------|
| A3-Mega   | NVIDIA H100 80GB | `hcc-a3m` |
| A3-Ultra  | NVIDIA H200 141GB | `hcc-a3u` |
| A3H       | NVIDIA H100 80GB  | `hcc-a3h` |
| A4        | NVIDIA B200 180GB | `hcc-a4`  |

> **Tip:** Use `sinfo` on the login node after connecting to verify available Slurm partitions and their current state.

---

## Step 2: Connect to the Cluster Login Node

Once the cluster is provisioned, find the SSH command by navigating to the [Google Compute Engine VM page](https://console.cloud.google.com/compute/instances) in the Google Cloud Console and clicking **SSH > View Google Cloud CLI command**.

The command will look similar to:

```
gcloud compute ssh --zone <zone> <login-node-name> --project <project-id>
```

All subsequent steps are run from this login node.

---

## Step 3: Configure Authentication

The training job requires two optional credentials:

| Credential | Purpose | Required? |
|------------|---------|----------|
| `HF_TOKEN` | Pull gated models/datasets from [Hugging Face](https://huggingface.co/settings/tokens) | Only for gated models |
| `WANDB_API_KEY` | Track experiments in [Weights & Biases](https://wandb.ai/authorize) | Optional (can be disabled) |

Edit the authentication file on the login node:

```
$HOME/vertex-oss-training/nemo_rl/configs/auth.sh
```

Set your tokens in that file. If you do not want to use W&B, you can disable it at launch time by setting `logger.wandb_enabled=false` in the `sbatch` command (shown in [Step 6](#step-6-launch-the-training-job)).

---

## Step 4: Download the Container Image

A prebuilt [Enroot](https://github.com/NVIDIA/enroot) `.sqsh` container image is provided for each region. Copy it into your working directory on the cluster's shared filesystem.

Available regions: `us`, `asia`, `europe`

The image path follows this pattern:

```
gs://vmds-containers-<region>/vmds_nemo_rl_squashfs/nemo_rl-h20260302.sqsh
```

Copy it into your launch directory:

```
$HOME/vertex-oss-training/nemo_rl/nemo_rl-h20260302.sqsh
```

> Alternatively, you can reference the container image directly from GCS without downloading it — set the image path in `launch.sh` to the GCS URI.

See the [Use prebuilt Docker image](https://cloud.google.com/vertex-ai/docs/training/training-clusters/nemo-rl) section of the official docs for more details.

---

## Step 5: Clone the NeMo RL Repository

The training scripts depend on the [NVIDIA NeMo RL](https://github.com/NVIDIA-NeMo/RL) open-source repository. Clone it into your working directory **with submodules**:

```bash
cd $HOME/vertex-oss-training/nemo_rl
git clone https://github.com/NVIDIA-NeMo/RL --recursive
```

> If you previously cloned without `--recursive`, run `git submodule update --init --recursive` inside the `RL/` directory.

The `launch.sh` script expects the NeMo RL code to be present at `$HOME/vertex-oss-training/nemo_rl/RL/` on the shared filesystem, since it mounts this directory into all Ray containers at runtime.

---

## Step 6: Launch the Training Job

Submit the GRPO training job using `sbatch`. The entrypoint is `launch.sh`, which:

1. Validates the container image and cluster type
2. Spins up a [Ray](https://docs.ray.io/) cluster across all allocated Slurm nodes
3. Executes `algorithms/dpo.sh` (the GRPO training script) in the Ray head container

### Single-partition clusters

```bash
cd $HOME/vertex-oss-training/nemo_rl
sbatch -N <num_nodes> launch.sh \
  --cluster_type <cluster_type> \
  --job_script algorithms/dpo.sh
```

### Multi-partition clusters

On clusters with multiple GPU types, pass `--partition` explicitly so Slurm schedules the job on the correct nodes:

```bash
sbatch -N 1 --partition=a3m launch.sh \
  --cluster_type hcc-a3m \
  --job_script algorithms/dpo.sh
```

> **`--cluster_type` vs `--partition`:** `--cluster_type` selects cluster-specific environment variables (NCCL paths, GPU counts). `--partition` tells Slurm which physical nodes to use. On multi-partition clusters both are required.

### Interactive mode

For debugging, you can launch an interactive session (no `--job_script` needed):

```bash
sbatch -N 1 launch.sh --cluster_type <cluster_type> --interactive
```

Once the job starts, an `attach.sh` script will be created in the job directory. Run it to open a shell inside the Ray head container with the full environment pre-configured.

### Key training parameters (editable in `algorithms/dpo.sh`)

| Parameter | Default | Description |
|-----------|---------|-------------|
| `--config` | `grpo-nano-v2-12b-1n8g-megatron.yaml` | GRPO recipe config file |
| `cluster.num_nodes` | from Slurm | Number of training nodes |
| `cluster.gpus_per_node` | 8 | GPUs per node |
| `logger.wandb_enabled` | `true` | Enable W&B logging |
| `logger.mlflow_enabled` | `false` | Enable MLflow logging |

See all available config options in the NeMo RL repository under [`examples/configs/recipes/`](https://github.com/NVIDIA-NeMo/RL/tree/main/examples/configs/recipes).

---

## Step 7: Monitor the Job

After submission, Slurm creates a directory named after the **Job ID** in your working directory. It contains:

```
<SLURM_JOB_ID>/
├── checkpoints/          # Model checkpoints (mounted inside container)
├── ray-logs/             # Ray head and worker logs
│   ├── ray-head.log
│   ├── ray-worker-*.log
│   └── ray-driver.log
├── nemo_rl_output.log    # Main Slurm job output
└── attach.sh             # (Interactive mode only) Shell attach script
```

### Useful monitoring commands

| Task | Command |
|------|---------|
| Check job queue | `squeue -u $USER` |
| View Slurm output | `tail -f <JOB_ID>/nemo_rl_output.log` |
| View Ray driver log | `tail -f <JOB_ID>/ray-logs/ray-driver.log` |
| Cancel a job | `scancel <JOB_ID>` |

### W&B Dashboard

If W&B logging is enabled, training metrics are streamed to your [Weights & Biases](https://wandb.ai) project in real time. The run is named `grpo-nanov2-12b-<JOB_ID>`.

---

## Reference Links

| Resource | Link |
|----------|------|
| Vertex AI Training Clusters overview | [cloud.google.com/vertex-ai/docs/training/training-clusters/overview](https://cloud.google.com/vertex-ai/docs/training/training-clusters/overview) |
| Create a cluster | [cloud.google.com/vertex-ai/docs/training/training-clusters/create-cluster](https://cloud.google.com/vertex-ai/docs/training/training-clusters/create-cluster) |
| NeMo RL on VTC (official guide) | [cloud.google.com/vertex-ai/docs/training/training-clusters/nemo-rl](https://cloud.google.com/vertex-ai/docs/training/training-clusters/nemo-rl) |
| NVIDIA NeMo RL GitHub | [github.com/NVIDIA-NeMo/RL](https://github.com/NVIDIA-NeMo/RL) |
| NeMo RL GRPO recipe configs | [github.com/NVIDIA-NeMo/RL/tree/main/examples/configs/recipes](https://github.com/NVIDIA-NeMo/RL/tree/main/examples/configs/recipes) |
| Hugging Face token settings | [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) |
| Weights & Biases API key | [wandb.ai/authorize](https://wandb.ai/authorize) |
| Ray documentation | [docs.ray.io](https://docs.ray.io) |
| Enroot (container runtime) | [github.com/NVIDIA/enroot](https://github.com/NVIDIA/enroot) |